# Plot vanilla CAN BUS vs Command Fix

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
from glob import glob
from collections import defaultdict

def decode_float_directions_to_str(float_direction):
    if float_direction == 1.0:
        return 'TURN LEFT'
    elif float_direction == 2.0:
        return 'TURN RIGHT'
    elif float_direction == 3.0:
        return 'GO STRAIGHT'
    elif float_direction == 4.0:
        return 'FOLLOW LANE'
    elif float_direction == 5.0:
        return 'CHANGELANE LEFT'
    elif float_direction == 6.0:
        return 'CHANGELANE RIGHT'
    else:
        raise ValueError("Unexpected direction identified %s" % float_direction)

def collect_trajectory_data(directory):
    """
    Collect trajectory data from all CAN bus files in the directory.
    Returns two dictionaries containing data for original and CMD fix files.
    """
    # Find all original CAN bus files and their corresponding CMD fix files
    can_bus_files = sorted(glob(os.path.join(directory, "can_bus*.json")))
    cmd_fix_files = sorted(glob(os.path.join(directory, "cmd_fix_can_bus*.json")))
    
    # Verify matching files
    assert len(can_bus_files) == len(cmd_fix_files), "Mismatch in number of files"
    
    # Initialize data structures to store trajectories
    can_bus_data = defaultdict(list)  # Will store data grouped by direction
    cmd_fix_data = defaultdict(list)
    
    # Process all file pairs
    for can_file, cmd_file in zip(can_bus_files, cmd_fix_files):
        # Extract sequence number to verify matching files
        can_seq = int(can_file.split('bus')[-1].split('.')[0])
        cmd_seq = int(cmd_file.split('bus')[-1].split('.')[0])
        assert can_seq == cmd_seq, f"Sequence mismatch: {can_seq} vs {cmd_seq}"
        
        # Read both files
        with open(can_file, 'r') as f:
            can_datum = json.load(f)
        with open(cmd_file, 'r') as f:
            cmd_datum = json.load(f)
            
        # Store position and direction for original CAN bus data
        if isinstance(can_datum, list):  # Handle both single-entry and list formats
            for entry in can_datum:
                direction = entry['direction']
                position = entry['ego_location'][:2]  # Only x,y coordinates
                can_bus_data[direction].append(position)
        else:
            direction = can_datum['direction']
            position = can_datum['ego_location'][:2]
            can_bus_data[direction].append(position)
            
        # Store position and direction for CMD fix data
        if isinstance(cmd_datum, list):
            for entry in cmd_datum:
                direction = entry['direction']
                position = entry['ego_location'][:2]
                cmd_fix_data[direction].append(position)
        else:
            direction = cmd_datum['direction']
            position = cmd_datum['ego_location'][:2]
            cmd_fix_data[direction].append(position)
    
    return can_bus_data, cmd_fix_data

def plot_trajectories(can_bus_data, cmd_fix_data):
    """
    Plot the trajectories with different colors for different directions.
    """
    direction_to_color = {
        1.0: 'red',      # TURN LEFT
        2.0: 'blue',     # TURN RIGHT
        3.0: 'green',    # GO STRAIGHT
        4.0: 'purple',   # FOLLOW LANE
        5.0: 'orange',   # CHANGELANE LEFT
        6.0: 'brown'     # CHANGELANE RIGHT
    }
    
    plt.figure(figsize=(12, 8))
    
    # Plot original CAN bus data
    for direction, positions in can_bus_data.items():
        positions = np.array(positions)
        if len(positions) > 0:
            plt.plot(positions[:, 0], positions[:, 1], 
                    'o-', color=direction_to_color[direction], alpha=0.5,
                    label=f'Original: {decode_float_directions_to_str(direction)}',
                    markersize=2)
    
    # Plot CMD fix data
    for direction, positions in cmd_fix_data.items():
        positions = np.array(positions)
        if len(positions) > 0:
            plt.plot(positions[:, 0], positions[:, 1], 
                    's--', color=direction_to_color[direction], alpha=0.5,
                    label=f'CMD Fix: {decode_float_directions_to_str(direction)}',
                    markersize=2)
    
    plt.title('Complete Trajectory Comparison (All Files)')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.axis('equal')
    plt.tight_layout()
    
    return plt.gcf()